# ProbKnot
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import tarfile
import subprocess
from pathlib import Path
from tqdm import tqdm

In [ ]:
method_name = "ProbKnot"
base = Path.cwd()

ProbKnot is included in the `RNAstructure` package.

In [ ]:
install_dir = base.parent / 'tools'
os.makedirs(install_dir, exist_ok=True)
RNAstructure_archive = install_dir / 'RNAstructureLinuxTextInterfaces64bit.tgz'
RNAstructure_path = install_dir / 'RNAstructure'

if not RNAstructure_path.exists() or not (RNAstructure_path / 'exe' / 'Fold').exists():
    %cd {install_dir}
    if not RNAstructure_archive.exists():
        !wget -q http://rna.urmc.rochester.edu/Releases/current/RNAstructureLinuxTextInterfaces64bit.tgz
    !tar xfz RNAstructureLinuxTextInterfaces64bit.tgz
else:
    print('RNAstructure already installed at', RNAstructure_path)

In [ ]:
source_path  = install_dir / "RNAstructure/"
data_table   = os.path.join(source_path, 'data_tables')
probknot_bin = os.path.join(source_path, 'exe', 'ProbKnot')

!{probknot_bin} --version

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(name, seq):
    tmp_fasta = f'ProbKnot_tmp_{name}.fasta'
    out_file_name = f'ProbKnot_clean_tmp_{name}.dot'

    with open(tmp_fasta, 'w') as ofile:
        ofile.write(f'>{name}\n{seq}\n')

    os.system(f"export DATAPATH={data_table}; {probknot_bin} {tmp_fasta} tmp.cs --sequence >/dev/null 2>&1")
    os.system(f"python ct2dot.py tmp.cs {out_file_name} -f full -q")

    if os.path.exists(tmp_fasta):
        os.remove(tmp_fasta)
    if os.path.exists('tmp.cs'):
        os.remove('tmp.cs')

    return out_file_name

In [ ]:
output_dir = Path("../prediction")
output_dir.mkdir(exist_ok=True)

out_fasta_name = output_dir / (method_name + ".fasta")

# Remove existing file if it exists
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    dot_file_name = run_folding(vid, seq)

    # Concatenate outputs only if folding was successful
    if dot_file_name and os.path.exists(dot_file_name):
        os.system(f"cat {dot_file_name} >> {out_fasta_name}")
        os.remove(dot_file_name)

    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s")

print(f"\nProcessing complete. Results saved to {out_fasta_name}")